<a href="https://colab.research.google.com/github/chithuapham/mse-python-ml-learning/blob/main/AI_LAP02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact_manual
from IPython.display import display

# ==========================================
# 1. Định nghĩa Mạng Sinh (Generator)
# ==========================================
class Generator(nn.Module):
    def __init__(self, latent_dim):
        super(Generator, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Linear(16, 2) # Đầu ra là tọa độ (x, y) 2D
        )

    def forward(self, x):
        return self.net(x)

# ==========================================
# 2. Định nghĩa Mạng Phân biệt (Discriminator)
# ==========================================
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid() # Đầu ra là xác suất (0 đến 1)
        )

    def forward(self, x):
        return self.net(x)

# ==========================================
# 3. Hàm tạo dữ liệu thật (Hình Parabol)
# ==========================================
def generate_real_data(batch_size):
    # Lấy x ngẫu nhiên từ -2 đến 2
    x = torch.rand(batch_size, 1) * 4 - 2
    # y = x^2 + nhiễu nhỏ
    y = x ** 2 + 0.1 * torch.randn(batch_size, 1)
    return torch.cat((x, y), dim=1)

# ==========================================
# 4. Hàm Huấn luyện và Trực quan hóa chính
# ==========================================
def train_gan_and_plot(lr_g, lr_d, epochs, batch_size, latent_dim):
    print(f"Đang tiến hành huấn luyện với {epochs} epochs...")

    # Khởi tạo mô hình
    G = Generator(latent_dim)
    D = Discriminator()

    # Khởi tạo hàm mất mát và bộ tối ưu
    criterion = nn.BCELoss()
    optimizer_G = optim.Adam(G.parameters(), lr=lr_g)
    optimizer_D = optim.Adam(D.parameters(), lr=lr_d)

    losses_G = []
    losses_D = []

    for epoch in range(epochs):
        # --- Huấn luyện Discriminator ---
        D.zero_grad()

        # Dữ liệu thật
        real_data = generate_real_data(batch_size)
        real_labels = torch.ones((batch_size, 1))
        output_real = D(real_data)
        loss_D_real = criterion(output_real, real_labels)

        # Dữ liệu giả (từ Generator)
        noise = torch.randn((batch_size, latent_dim))
        fake_data = G(noise)
        fake_labels = torch.zeros((batch_size, 1))
        output_fake = D(fake_data.detach()) # Dùng detach để không tính gradient ngược về G
        loss_D_fake = criterion(output_fake, fake_labels)

        # Cập nhật Discriminator
        loss_D = loss_D_real + loss_D_fake
        loss_D.backward()
        optimizer_D.step()

        # --- Huấn luyện Generator ---
        G.zero_grad()

        # Tạo dữ liệu giả mới để đánh lừa D (G muốn D dự đoán là 1 - thật)
        noise = torch.randn((batch_size, latent_dim))
        fake_data = G(noise)
        output_fake_for_G = D(fake_data)

        loss_G = criterion(output_fake_for_G, real_labels) # Nhãn là 1 (thật) vì G muốn lừa D
        loss_G.backward()
        optimizer_G.step()

        # Lưu loss để vẽ biểu đồ
        if epoch % 10 == 0:
            losses_G.append(loss_G.item())
            losses_D.append(loss_D.item())

    print("Huấn luyện hoàn tất. Đang tạo biểu đồ...")

    # --- Trực quan hóa Kết quả ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # 1. Biểu đồ Loss
    ax1.plot(losses_G, label='Generator Loss', color='orange')
    ax1.plot(losses_D, label='Discriminator Loss', color='blue')
    ax1.set_title('Đồ thị Mất mát (Loss) qua các Epoch')
    ax1.set_xlabel('Epoch (x10)')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True, linestyle='--', alpha=0.6)

    # 2. Biểu đồ Phân phối dữ liệu
    # Sinh một lượng lớn điểm để kiểm tra
    with torch.no_grad():
        test_noise = torch.randn((500, latent_dim))
        generated_points = G(test_noise).numpy()

    real_points = generate_real_data(500).numpy()

    ax2.scatter(real_points[:, 0], real_points[:, 1], color='blue', alpha=0.5, label='Dữ liệu Thật', s=15)
    ax2.scatter(generated_points[:, 0], generated_points[:, 1], color='red', alpha=0.5, label='Dữ liệu GAN sinh ra', s=15)
    ax2.set_title('Phân phối Dữ liệu (Thật vs Giả)')
    ax2.set_xlim(-3, 3)
    ax2.set_ylim(-1, 5)
    ax2.legend()
    ax2.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()

# ==========================================
# 5. Giao diện người dùng tương tác (Widgets)
# ==========================================
print("Hãy điều chỉnh các thanh trượt bên dưới và nhấn 'Run Interact' để xem sự thay đổi.")

# Sử dụng interact_manual để chỉ chạy khi người dùng bấm nút
interact_manual(
    train_gan_and_plot,
    lr_g=widgets.FloatLogSlider(value=0.01, base=10, min=-4, max=-1, step=0.1, description='LR Generator'),
    lr_d=widgets.FloatLogSlider(value=0.01, base=10, min=-4, max=-1, step=0.1, description='LR Discriminator'),
    epochs=widgets.IntSlider(value=800, min=100, max=3000, step=100, description='Epochs'),
    batch_size=widgets.IntSlider(value=64, min=16, max=256, step=16, description='Batch Size'),
    latent_dim=widgets.IntSlider(value=2, min=1, max=10, step=1, description='Latent Dim')
);

Hãy điều chỉnh các thanh trượt bên dưới và nhấn 'Run Interact' để xem sự thay đổi.


interactive(children=(FloatLogSlider(value=0.01, description='LR Generator', max=-1.0, min=-4.0), FloatLogSlid…